In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

RANDOM_STATE = 420

In [2]:
import glob

print(glob.glob("*.csv"))

['count_sanity_check_summary.csv', 'cleaned_housing_data.csv', 'removed_training_target_outliers.csv']


In [3]:
# ============================================================
# 1. Read and combine monthly CSV files
# ============================================================

files = [
    "../../CRMLSSold202505.csv",
    "../../CRMLSSold202506.csv",
    "../../CRMLSSold202507.csv",
    "../../CRMLSSold202508.csv",
    "../../CRMLSSold202509.csv",
    "../../CRMLSSold202510.csv",
    "../../CRMLSSold202511.csv",
    "../../CRMLSSold202512.csv",
    "../../CRMLSSold202601.csv",
    "../../CRMLSSold202602.csv",
    "../../CRMLSSold202603.csv",
    "../../CRMLSSold202604.csv",
    "../../CRMLSSold202605.csv",
    "../../CRMLSSold202606.csv"
]

dataframes = []

for file in files:
    monthly_df = pd.read_csv(
        file,
        low_memory=False
    )

    monthly_df["SourceFile"] = file
    dataframes.append(monthly_df)

df = pd.concat(
    dataframes,
    ignore_index=True
)

print("=" * 70)
print("RAW COMBINED DATA")
print("=" * 70)
print("Shape:", df.shape)
print("Number of files:", len(files))

RAW COMBINED DATA
Shape: (306330, 79)
Number of files: 14


In [4]:
# ============================================================
# 2. Keep residential single-family properties
# ============================================================

required_property_cols = [
    "PropertyType",
    "PropertySubType"
]

missing_required_property_cols = [
    col for col in required_property_cols
    if col not in df.columns
]

if missing_required_property_cols:
    raise KeyError(
        f"Missing required property columns: "
        f"{missing_required_property_cols}"
    )

df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

print("\n")
print("=" * 70)
print("PROPERTY FILTER")
print("=" * 70)
print("Shape after keeping Residential SingleFamilyResidence:", df.shape)


# ============================================================
# 3. Remove duplicated listings
# ============================================================

if "ListingKey" not in df.columns:
    raise KeyError("ListingKey is required for duplicate removal.")

duplicates_before = df["ListingKey"].duplicated().sum()

print("\n")
print("=" * 70)
print("DUPLICATE CHECK")
print("=" * 70)
print("Duplicated ListingKey values before cleaning:", duplicates_before)

df = df.drop_duplicates(
    subset="ListingKey",
    keep="last"
).copy()

duplicates_after = df["ListingKey"].duplicated().sum()

print("Duplicated ListingKey values after cleaning:", duplicates_after)
print("Shape after duplicate removal:", df.shape)


# ============================================================
# 4. Initial missing-value summary
# ============================================================

missing_summary_initial = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

missing_summary_initial = missing_summary_initial.sort_values(
    "missing_percent",
    ascending=False
)

print("\n")
print("=" * 70)
print("INITIAL MISSING-VALUE SUMMARY")
print("=" * 70)

display(
    missing_summary_initial[
        missing_summary_initial["missing_count"] > 0
    ]
)





PROPERTY FILTER
Shape after keeping Residential SingleFamilyResidence: (154855, 79)


DUPLICATE CHECK
Duplicated ListingKey values before cleaning: 115


Duplicated ListingKey values after cleaning: 0
Shape after duplicate removal: (154740, 79)




INITIAL MISSING-VALUE SUMMARY


,missing_count,missing_percent
CoveredSpaces,154740,100.000000
BusinessType,154740,100.000000
ElementarySchoolDistrict,154740,100.000000
MiddleOrJuniorSchoolDistrict,154740,100.000000
TaxAnnualAmount,154740,100.000000
AboveGradeFinishedArea,154740,100.000000
FireplacesTotal,154740,100.000000
TaxYear,154740,100.000000
WaterfrontYN,154656,99.945715
BelowGradeFinishedArea,153634,99.285253


In [5]:
# ============================================================
# 5. Convert important numerical columns
# ============================================================
# Some CSV files may store numerical values as strings.
# Converting them here makes the sanity checks more reliable.

numeric_candidates = [
    "Latitude",
    "Longitude",
    "LotSizeSquareFeet",
    "LivingArea",
    "ParkingTotal",
    "GarageSpaces",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "MainLevelBedrooms",
    "AssociationFee",
    "Stories",
    "YearBuilt",
    "ClosePrice"
]

for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


In [6]:
# ============================================================
# 6. Basic invalid-value checks
# ============================================================
"""
These rules identify values that are impossible or clearly invalid.

Feature values are converted to NaN instead of deleting the full row.
The missing values will later be handled using training-set information.
"""


def replace_invalid_values(
    dataframe,
    column,
    invalid_mask,
    rule_description
):
    """
    Replace invalid values in one column with NaN and print a summary.
    """

    if column not in dataframe.columns:
        print(f"{column}: column not found, skipped.")
        return dataframe

    missing_before = dataframe[column].isna().sum()
    invalid_count = invalid_mask.fillna(False).sum()

    dataframe.loc[
        invalid_mask.fillna(False),
        column
    ] = np.nan

    missing_after = dataframe[column].isna().sum()

    print(f"\n{column}")
    print("-" * 50)
    print("Rule:", rule_description)
    print("Values converted to NaN:", invalid_count)
    print(
        "Additional missing values:",
        missing_after - missing_before
    )

    return dataframe


print("\n")
print("=" * 70)
print("BASIC INVALID-VALUE CHECKS")
print("=" * 70)


if "Latitude" in df.columns:
    df = replace_invalid_values(
        dataframe=df,
        column="Latitude",
        invalid_mask=(
            (df["Latitude"] < 32) |
            (df["Latitude"] > 42)
        ),
        rule_description="Latitude must be between 32 and 42."
    )


if "Longitude" in df.columns:
    df = replace_invalid_values(
        dataframe=df,
        column="Longitude",
        invalid_mask=(
            (df["Longitude"] < -125) |
            (df["Longitude"] > -114)
        ),
        rule_description="Longitude must be between -125 and -114."
    )


if "LotSizeSquareFeet" in df.columns:
    df = replace_invalid_values(
        dataframe=df,
        column="LotSizeSquareFeet",
        invalid_mask=df["LotSizeSquareFeet"] <= 100,
        rule_description="LotSizeSquareFeet must be greater than 100."
    )


if "LivingArea" in df.columns:
    df = replace_invalid_values(
        dataframe=df,
        column="LivingArea",
        invalid_mask=df["LivingArea"] <= 0,
        rule_description="LivingArea must be positive."
    )


if "ParkingTotal" in df.columns:
    df = replace_invalid_values(
        dataframe=df,
        column="ParkingTotal",
        invalid_mask=df["ParkingTotal"] < 0,
        rule_description="ParkingTotal cannot be negative."
    )


if "GarageSpaces" in df.columns:
    df = replace_invalid_values(
        dataframe=df,
        column="GarageSpaces",
        invalid_mask=df["GarageSpaces"] < 0,
        rule_description="GarageSpaces cannot be negative."
    )


if "BathroomsTotalInteger" in df.columns:
    df = replace_invalid_values(
        dataframe=df,
        column="BathroomsTotalInteger",
        invalid_mask=df["BathroomsTotalInteger"] <= 0,
        rule_description="BathroomsTotalInteger must be positive."
    )


if "BedroomsTotal" in df.columns:
    df = replace_invalid_values(
        dataframe=df,
        column="BedroomsTotal",
        invalid_mask=df["BedroomsTotal"] <= 0,
        rule_description="BedroomsTotal must be positive."
    )


if "MainLevelBedrooms" in df.columns:
    df = replace_invalid_values(
        dataframe=df,
        column="MainLevelBedrooms",
        invalid_mask=df["MainLevelBedrooms"] < 0,
        rule_description="MainLevelBedrooms cannot be negative."
    )


if "AssociationFee" in df.columns:
    df = replace_invalid_values(
        dataframe=df,
        column="AssociationFee",
        invalid_mask=df["AssociationFee"] < 0,
        rule_description="AssociationFee cannot be negative."
    )


# MainLevelBedrooms should not exceed BedroomsTotal
if (
    "MainLevelBedrooms" in df.columns and
    "BedroomsTotal" in df.columns
):
    df = replace_invalid_values(
        dataframe=df,
        column="MainLevelBedrooms",
        invalid_mask=(
            df["MainLevelBedrooms"] >
            df["BedroomsTotal"]
        ),
        rule_description=(
            "MainLevelBedrooms cannot be greater than BedroomsTotal."
        )
    )


print("\nNumerical summary after basic invalid-value checks:")
display(df.describe(include=[np.number]).T)





BASIC INVALID-VALUE CHECKS

Latitude
--------------------------------------------------
Rule: Latitude must be between 32 and 42.
Values converted to NaN: 30
Additional missing values: 30

Longitude
--------------------------------------------------
Rule: Longitude must be between -125 and -114.
Values converted to NaN: 43
Additional missing values: 43

LotSizeSquareFeet
--------------------------------------------------
Rule: LotSizeSquareFeet must be greater than 100.
Values converted to NaN: 389
Additional missing values: 389

LivingArea
--------------------------------------------------
Rule: LivingArea must be positive.
Values converted to NaN: 70
Additional missing values: 70

ParkingTotal
--------------------------------------------------
Rule: ParkingTotal cannot be negative.
Values converted to NaN: 23
Additional missing values: 23

GarageSpaces
--------------------------------------------------
Rule: GarageSpaces cannot be negative.
Values converted to NaN: 0
Additional mis

,count,mean,std,min,25%,50%,75%,max
OriginalListPrice,154417.0,1.388516e+06,8.041270e+06,0.000000e+00,6.390000e+05,8.990000e+05,1.449990e+06,1.302000e+09
ListingKey,154740.0,1.132494e+09,1.972514e+07,4.217759e+08,1.114574e+09,1.132305e+09,1.151480e+09,1.176283e+09
ClosePrice,154740.0,1.341825e+06,7.757042e+06,0.000000e+00,6.250000e+05,8.950000e+05,1.425000e+06,9.895000e+08
Latitude,154697.0,3.474211e+01,1.700309e+00,3.254525e+01,3.376107e+01,3.408373e+01,3.484811e+01,4.189471e+01
Longitude,154684.0,-1.186417e+02,1.847537e+00,-1.241932e+02,-1.191626e+02,-1.180329e+02,-1.172602e+02,-1.143472e+02
LivingArea,154586.0,2.051946e+03,1.046055e+03,1.000000e+02,1.389000e+03,1.823000e+03,2.444000e+03,5.650000e+04
ListPrice,154740.0,1.270621e+06,1.608576e+06,8.000000e+03,6.250000e+05,8.980000e+05,1.400000e+06,1.375000e+08
DaysOnMarket,154740.0,3.934682e+01,5.278184e+01,-2.650000e+02,8.000000e+00,2.000000e+01,5.100000e+01,2.177000e+03
FireplacesTotal,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AboveGradeFinishedArea,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# ============================================================
# 7. Count-based feature sanity checks
# ============================================================
"""
Count-based features use broad manually defined domain checks rather than
IQR-based thresholds.

These variables are discrete counts and are often concentrated around small
integer values. IQR-based rules may incorrectly flag legitimate large homes.

Values outside the manually defined range are converted to NaN rather than
causing the entire property row to be removed.

Because these are manually defined domain rules, they can be applied before
the train/test split.
"""


def apply_count_sanity_rule(
    dataframe,
    column,
    lower_bound,
    upper_bound
):
    """
    Convert unrealistic count-based values to NaN.
    """

    if column not in dataframe.columns:
        print(f"{column}: column not found, skipped.")
        return dataframe, None

    dataframe[column] = pd.to_numeric(
        dataframe[column],
        errors="coerce"
    )

    missing_before = dataframe[column].isna().sum()
    minimum_before = dataframe[column].min()
    maximum_before = dataframe[column].max()

    below_mask = dataframe[column] < lower_bound
    above_mask = dataframe[column] > upper_bound
    invalid_mask = below_mask | above_mask

    rows_below = below_mask.sum()
    rows_above = above_mask.sum()

    dataframe.loc[
        invalid_mask,
        column
    ] = np.nan

    missing_after = dataframe[column].isna().sum()

    summary = {
        "Feature": column,
        "LowerBound": lower_bound,
        "UpperBound": upper_bound,
        "MinimumBefore": minimum_before,
        "MaximumBefore": maximum_before,
        "RowsBelowLowerBound": rows_below,
        "RowsAboveUpperBound": rows_above,
        "ValuesConvertedToNaN": (
            missing_after - missing_before
        )
    }

    print(f"\n{column}")
    print("-" * 50)
    print(f"Valid range: {lower_bound} to {upper_bound}")
    print("Minimum before cleaning:", minimum_before)
    print("Maximum before cleaning:", maximum_before)
    print("Rows below lower bound:", rows_below)
    print("Rows above upper bound:", rows_above)
    print(
        "Values converted to NaN:",
        missing_after - missing_before
    )

    return dataframe, summary


count_sanity_rules = {
    "ParkingTotal": {
        "lower_bound": 0,
        "upper_bound": 50
    },
    "GarageSpaces": {
        "lower_bound": 0,
        "upper_bound": 20
    },
    "BedroomsTotal": {
        "lower_bound": 1,
        "upper_bound": 20
    },
    "BathroomsTotalInteger": {
        "lower_bound": 1,
        "upper_bound": 20
    },
    "MainLevelBedrooms": {
        "lower_bound": 0,
        "upper_bound": 20
    }
}


print("\n")
print("=" * 70)
print("COUNT-BASED SANITY CHECKS")
print("=" * 70)

count_cleaning_records = []

for col, rule in count_sanity_rules.items():

    df, summary = apply_count_sanity_rule(
        dataframe=df,
        column=col,
        lower_bound=rule["lower_bound"],
        upper_bound=rule["upper_bound"]
    )

    if summary is not None:
        count_cleaning_records.append(summary)


count_cleaning_summary = pd.DataFrame(
    count_cleaning_records
)

print("\nCount-based sanity-check summary:")
display(count_cleaning_summary)

print("Shape after count-based sanity checks:", df.shape)




COUNT-BASED SANITY CHECKS

ParkingTotal
--------------------------------------------------
Valid range: 0 to 50
Minimum before cleaning: 0.0
Maximum before cleaning: 15720.0
Rows below lower bound: 0
Rows above upper bound: 76
Values converted to NaN: 76

GarageSpaces
--------------------------------------------------
Valid range: 0 to 20
Minimum before cleaning: 0.0
Maximum before cleaning: 600.0
Rows below lower bound: 0
Rows above upper bound: 19
Values converted to NaN: 19

BedroomsTotal
--------------------------------------------------
Valid range: 1 to 20
Minimum before cleaning: 1.0
Maximum before cleaning: 22.0
Rows below lower bound: 0
Rows above upper bound: 1
Values converted to NaN: 1

BathroomsTotalInteger
--------------------------------------------------
Valid range: 1 to 20
Minimum before cleaning: 1.0
Maximum before cleaning: 35.0
Rows below lower bound: 0
Rows above upper bound: 5
Values converted to NaN: 5

MainLevelBedrooms
---------------------------------------

,Feature,LowerBound,UpperBound,MinimumBefore,MaximumBefore,RowsBelowLowerBound,RowsAboveUpperBound,ValuesConvertedToNaN
0,ParkingTotal,0,50,0.0,15720.0,0,76,76
1,GarageSpaces,0,20,0.0,600.0,0,19,19
2,BedroomsTotal,1,20,1.0,22.0,0,1,1
3,BathroomsTotalInteger,1,20,1.0,35.0,0,5,5
4,MainLevelBedrooms,0,20,0.0,16.0,0,0,0


Shape after count-based sanity checks: (154740, 79)


In [8]:
# ============================================================
# 8. Missing-value summary after sanity checks
# ============================================================

missing_summary_after_sanity = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

missing_summary_after_sanity = (
    missing_summary_after_sanity[
        missing_summary_after_sanity["missing_count"] > 0
    ]
    .sort_values(
        "missing_percent",
        ascending=False
    )
)

print("\n")
print("=" * 70)
print("MISSING VALUES AFTER SANITY CHECKS")
print("=" * 70)

display(missing_summary_after_sanity)


# ============================================================
# 9. Drop columns with more than 70% missing values
# ============================================================

missing_percent = df.isna().mean() * 100

high_missing_cols = missing_percent[
    missing_percent > 70
].index.tolist()

# Protect essential columns in case they unexpectedly have high missingness.
protected_columns = [
    "ListingKey",
    "ClosePrice",
    "CloseDate",
    "LivingArea",
    "Latitude",
    "Longitude"
]

high_missing_cols = [
    col for col in high_missing_cols
    if col not in protected_columns
]

print("\n")
print("=" * 70)
print("HIGH-MISSINGNESS COLUMN REMOVAL")
print("=" * 70)

print("Number of columns removed:", len(high_missing_cols))
print(high_missing_cols)

df = df.drop(
    columns=high_missing_cols,
    errors="ignore"
)

print("Remaining columns:", df.shape[1])


# ============================================================
# 10. Remove personal, agent, and office information
# ============================================================

personal_keywords = [
    "agent",
    "office",
    "email",
    "mlsid",
    "name"
]

personal_cols = [
    col for col in df.columns
    if any(
        keyword in col.lower()
        for keyword in personal_keywords
    )
]

print("\n")
print("=" * 70)
print("PERSONAL-INFORMATION COLUMN REMOVAL")
print("=" * 70)

print("Personal or agent-related columns:")
print(personal_cols)

df = df.drop(
    columns=personal_cols,
    errors="ignore"
)

print("Remaining columns:", df.shape[1])


# ============================================================
# 11. Remove redundant lot-size variables
# ============================================================

drop_lot_cols = [
    "LotSizeAcres",
    "LotSizeArea"
]

existing_lot_cols = [
    col for col in drop_lot_cols
    if col in df.columns
]

df = df.drop(
    columns=existing_lot_cols,
    errors="ignore"
)

print("\n")
print("=" * 70)
print("LOT-SIZE FEATURE SELECTION")
print("=" * 70)

print("Using LotSizeSquareFeet as the lot-size feature.")
print("Dropped:", existing_lot_cols)


# ============================================================
# 12. Remove target-leakage columns
# ============================================================

leakage_columns = [
    "ListPrice",
    "OriginalListPrice"
]

existing_leakage_cols = [
    col for col in leakage_columns
    if col in df.columns
]

df = df.drop(
    columns=existing_leakage_cols,
    errors="ignore"
)

print("\n")
print("=" * 70)
print("TARGET-LEAKAGE COLUMN REMOVAL")
print("=" * 70)

print("Dropped leakage columns:", existing_leakage_cols)


# ============================================================
# 13. Convert and validate CloseDate
# ============================================================

if "CloseDate" not in df.columns:
    raise KeyError("CloseDate is required for the time-based split.")

df["CloseDate"] = pd.to_datetime(
    df["CloseDate"],
    errors="coerce"
)

invalid_close_dates = df["CloseDate"].isna().sum()

print("\n")
print("=" * 70)
print("CLOSE DATE VALIDATION")
print("=" * 70)

print("Rows with invalid or missing CloseDate:", invalid_close_dates)

# A time split cannot be performed without CloseDate.
df = df.dropna(
    subset=["CloseDate"]
).copy()

df = df.sort_values(
    "CloseDate"
).reset_index(drop=True)


# ============================================================
# 14. Validate the target variable
# ============================================================

if "ClosePrice" not in df.columns:
    raise KeyError("ClosePrice is required as the target variable.")

df["ClosePrice"] = pd.to_numeric(
    df["ClosePrice"],
    errors="coerce"
)

invalid_target_mask = (
    df["ClosePrice"].isna() |
    (df["ClosePrice"] <= 0)
)

print("\n")
print("=" * 70)
print("TARGET VALIDITY CHECK")
print("=" * 70)

print(
    "Rows with missing, zero, or negative ClosePrice:",
    invalid_target_mask.sum()
)

# ClosePrice cannot be imputed because it is the prediction target.
df = df.loc[
    ~invalid_target_mask
].copy()

print("Shape after target validity check:", df.shape)



MISSING VALUES AFTER SANITY CHECKS


,missing_count,missing_percent
AboveGradeFinishedArea,154740,100.000000
TaxAnnualAmount,154740,100.000000
FireplacesTotal,154740,100.000000
MiddleOrJuniorSchoolDistrict,154740,100.000000
CoveredSpaces,154740,100.000000
BusinessType,154740,100.000000
ElementarySchoolDistrict,154740,100.000000
TaxYear,154740,100.000000
WaterfrontYN,154656,99.945715
BelowGradeFinishedArea,153634,99.285253




HIGH-MISSINGNESS COLUMN REMOVAL
Number of columns removed: 22
['WaterfrontYN', 'BasementYN', 'CoListOfficeName', 'CoListAgentFirstName', 'CoListAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'ElementarySchool', 'BuilderName', 'TaxYear', 'BuildingAreaTotal', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'HighSchool', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict']
Remaining columns: 57


PERSONAL-INFORMATION COLUMN REMOVAL
Personal or agent-related columns:
['BuyerAgentAOR', 'ListAgentAOR', 'ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName', 'ListOfficeName', 'BuyerOfficeName', 'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'SubdivisionName', 'BuyerOfficeAOR']
Remaining columns: 44


LOT-SIZE FEATURE SELECTION
Using LotSizeSquareFeet as the lot-size feature.
Dropped: ['LotSizeAcres', 'L



TARGET VALIDITY CHECK
Rows with missing, zero, or negative ClosePrice: 1


Shape after target validity check: (154739, 40)


In [9]:
# ============================================================
# 15. Time-based train/test split
# ============================================================
"""
The most recent available month is used as the test set.

The six months immediately before the test month are used as the training set.
This matches the original six-month training-window design.
"""

df["YearMonth"] = df["CloseDate"].dt.to_period("M")

available_months = (
    df["YearMonth"]
    .value_counts()
    .sort_index()
)

print("\n")
print("=" * 70)
print("TIME-BASED TRAIN/TEST SPLIT")
print("=" * 70)

print("Available months:")
print(available_months)

latest_month = df["YearMonth"].max()

train_window = 6

train_months = pd.period_range(
    end=latest_month - 1,
    periods=train_window,
    freq="M"
)

train_df = df[
    df["YearMonth"].isin(train_months)
].copy()

test_df = df[
    df["YearMonth"] == latest_month
].copy()

if train_df.empty:
    raise ValueError(
        "The training set is empty. Check the available months "
        "or reduce train_window."
    )

if test_df.empty:
    raise ValueError(
        "The test set is empty. Check CloseDate and YearMonth."
    )

print("\nLatest month / test month:", latest_month)

print("\nTraining months:")
print(sorted(train_df["YearMonth"].unique()))

print("\nTest month:")
print(sorted(test_df["YearMonth"].unique()))

print("\nTrain shape before target outlier removal:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain date range:")
print(
    train_df["CloseDate"].min(),
    "to",
    train_df["CloseDate"].max()
)

print("\nTest date range:")
print(
    test_df["CloseDate"].min(),
    "to",
    test_df["CloseDate"].max()
)

# Add the split indicator before removing the temporary YearMonth column.
train_df["Dataset"] = "Train"
test_df["Dataset"] = "Test"

train_df = train_df.drop(
    columns=["YearMonth"]
)

test_df = test_df.drop(
    columns=["YearMonth"]
)





TIME-BASED TRAIN/TEST SPLIT
Available months:
YearMonth
2025-05    11762
2025-06    11692
2025-07    12110
2025-08    11443
2025-09    11449
2025-10    12021
2025-11     9720
2025-12    10448
2026-01     7485
2026-02     8542
2026-03    11169
2026-04    12026
2026-05    12014
2026-06    12858
Freq: M, Name: count, dtype: int64

Latest month / test month: 2026-06

Training months:
[Period('2025-12', 'M'), Period('2026-01', 'M'), Period('2026-02', 'M'), Period('2026-03', 'M'), Period('2026-04', 'M'), Period('2026-05', 'M')]

Test month:
[Period('2026-06', 'M')]

Train shape before target outlier removal: (61684, 41)
Test shape: (12858, 41)

Train date range:
2025-12-01 00:00:00 to 2026-05-31 00:00:00

Test date range:
2026-06-01 00:00:00 to 2026-06-30 00:00:00


In [10]:
# ============================================================
# 16. Handle target outliers in the training set only
# ============================================================
"""
ClosePrice outlier thresholds are calculated using only the training set.

PricePerLivingArea is used only for outlier inspection. It is removed before
modeling because it contains ClosePrice and would cause target leakage.

Low-end outliers:
    ClosePrice below the training-set 0.1st percentile.

High-end outliers:
    ClosePrice above the training-set 99.9th percentile
    AND
    PricePerLivingArea above the high-price subset's IQR cutoff.

The test set is not filtered using these outlier thresholds.
"""

if "LivingArea" not in train_df.columns:
    raise KeyError(
        "LivingArea is required for PricePerLivingArea inspection."
    )

train_df["LivingArea"] = pd.to_numeric(
    train_df["LivingArea"],
    errors="coerce"
)

train_df["PricePerLivingArea"] = np.nan

valid_living_area_mask = (
    train_df["LivingArea"].notna() &
    (train_df["LivingArea"] > 0)
)

train_df.loc[
    valid_living_area_mask,
    "PricePerLivingArea"
] = (
    train_df.loc[
        valid_living_area_mask,
        "ClosePrice"
    ] /
    train_df.loc[
        valid_living_area_mask,
        "LivingArea"
    ]
)

train_df["PricePerLivingArea"] = (
    train_df["PricePerLivingArea"]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


# Calculate thresholds using the training set only.
low_price_threshold = train_df["ClosePrice"].quantile(0.001)
high_price_threshold = train_df["ClosePrice"].quantile(0.99)

low_price_rows = train_df[
    train_df["ClosePrice"] < low_price_threshold
].copy()

high_price_rows = train_df[
    train_df["ClosePrice"] > high_price_threshold
].copy()


print("\n")
print("=" * 70)
print("TRAINING TARGET OUTLIER INSPECTION")
print("=" * 70)

print("Low ClosePrice threshold:", low_price_threshold)
print("High ClosePrice threshold:", high_price_threshold)

print("\nPricePerLivingArea for low-price observations:")
print(
    low_price_rows["PricePerLivingArea"].describe()
)

print("\nPricePerLivingArea for high-price observations:")
print(
    high_price_rows["PricePerLivingArea"].describe()
)


high_price_per_area = (
    high_price_rows["PricePerLivingArea"]
    .dropna()
)

if len(high_price_per_area) >= 4:

    high_area_q1 = high_price_per_area.quantile(0.25)
    high_area_q3 = high_price_per_area.quantile(0.75)
    high_area_iqr = high_area_q3 - high_area_q1

    if (
        pd.notna(high_area_iqr) and
        high_area_iqr > 0
    ):
        price_per_area_cutoff = (
            high_area_q3 +
            1.5 * high_area_iqr
        )
    else:
        price_per_area_cutoff = np.inf

else:
    high_area_q1 = np.nan
    high_area_q3 = np.nan
    high_area_iqr = np.nan
    price_per_area_cutoff = np.inf


print("\nHigh-price PricePerLivingArea Q1:", high_area_q1)
print("High-price PricePerLivingArea Q3:", high_area_q3)
print("High-price PricePerLivingArea IQR:", high_area_iqr)
print("PricePerLivingArea cutoff:", price_per_area_cutoff)


# Low total-price outliers.
extreme_low_mask = (
    train_df["ClosePrice"] <
    low_price_threshold
)

# A high-end row must have both extreme total price and extreme unit price.
extreme_high_mask = (
    (
        train_df["ClosePrice"] >
        high_price_threshold
    ) &
    (
        train_df["PricePerLivingArea"] >
        price_per_area_cutoff
    )
)

target_outlier_mask = (
    extreme_low_mask |
    extreme_high_mask
)


removed_target_outliers = train_df.loc[
    target_outlier_mask
].copy()

print("\nRows removed from training set at low end:")
print(extreme_low_mask.sum())

print("\nRows removed from training set at high end:")
print(extreme_high_mask.sum())

print("\nTotal target outliers removed from training:")
print(target_outlier_mask.sum())


inspection_cols = [
    col for col in [
        "ListingKey",
        "CloseDate",
        "ClosePrice",
        "LivingArea",
        "PricePerLivingArea",
        "City",
        "PostalCode",
        "MLSAreaMajor"
    ]
    if col in removed_target_outliers.columns
]

print("\nRemoved target-outlier observations:")
display(
    removed_target_outliers[
        inspection_cols
    ].sort_values("ClosePrice")
)


train_shape_before_outliers = train_df.shape

train_df = train_df.loc[
    ~target_outlier_mask
].copy()

# Drop the target-derived inspection variable.
train_df = train_df.drop(
    columns=["PricePerLivingArea"],
    errors="ignore"
)

test_df = test_df.drop(
    columns=["PricePerLivingArea"],
    errors="ignore"
)

print("\nTrain shape before target outlier removal:")
print(train_shape_before_outliers)

print("Train shape after target outlier removal:")
print(train_df.shape)

print("Test shape unchanged:")
print(test_df.shape)


# ============================================================
# 17. Drop redundant or unused raw columns
# ============================================================

drop_raw_cols = [
    "StreetNumberNumeric",
    "UnparsedAddress",
    # Temporary source tracking column.
    "SourceFile"
]

existing_drop_raw_cols = [
    col for col in drop_raw_cols
    if col in train_df.columns or col in test_df.columns
]

train_df = train_df.drop(
    columns=[
        col for col in drop_raw_cols
        if col in train_df.columns
    ]
)

test_df = test_df.drop(
    columns=[
        col for col in drop_raw_cols
        if col in test_df.columns
    ]
)

print("\n")
print("=" * 70)
print("REDUNDANT RAW COLUMN REMOVAL")
print("=" * 70)
print("Dropped:", existing_drop_raw_cols)




TRAINING TARGET OUTLIER INSPECTION
Low ClosePrice threshold: 85000.0
High ClosePrice threshold: 6500000.0

PricePerLivingArea for low-price observations:
count     58.000000
mean      58.701701
std       52.524528
min        0.000498
25%       25.711934
50%       50.930604
75%       72.514881
max      338.541667
Name: PricePerLivingArea, dtype: float64

PricePerLivingArea for high-price observations:
count       597.000000
mean       6731.634693
std       43788.442253
min         466.390719
25%        1329.734053
50%        1818.181818
75%        2397.743300
max      584239.130435
Name: PricePerLivingArea, dtype: float64

High-price PricePerLivingArea Q1: 1329.7340531893622
High-price PricePerLivingArea Q3: 2397.743300423131
High-price PricePerLivingArea IQR: 1068.009247233769
PricePerLivingArea cutoff: 3999.7571712737845

Rows removed from training set at low end:
58

Rows removed from training set at high end:
41

Total target outliers removed from training:
99

Removed target-outl

,ListingKey,CloseDate,ClosePrice,LivingArea,PricePerLivingArea,City,PostalCode,MLSAreaMajor
98087,1144223437,2026-01-30,1.750000e+00,3513.0,0.000498,Palm Desert,92211,324 - East Palm Desert
129288,1151459976,2026-04-29,6.850000e+02,2980.0,0.229866,Oakley,94561,NaN
99053,1147508351,2026-02-04,8.000000e+03,984.0,8.130081,Trona,93562,TRNA - Trona
94938,1110035161,2026-01-17,8.300000e+03,3886.0,2.135872,Altadena,91001,604 - Altadena
129916,1156473927,2026-05-01,1.190000e+04,4316.0,2.757183,Dana Point,92624,CB - Capistrano Beach
100770,1108190915,2026-02-11,1.800000e+04,1488.0,12.096774,Blythe,92225,374 - Blythe
98918,1149711163,2026-02-04,2.227500e+04,1587.0,14.035917,Victorville,92395,VIC - Victorville
134186,1111372750,2026-05-12,2.600000e+04,2206.0,11.786038,Trona,93562,TRNA - Trona
141781,1150901810,2026-05-29,2.680000e+04,1376.0,19.476744,Trona,93562,TRNA - Trona
116067,1151314793,2026-03-26,2.800000e+04,408.0,68.627451,Landers,92285,699 - Not Defined



Train shape before target outlier removal:
(61684, 42)
Train shape after target outlier removal:
(61585, 41)
Test shape unchanged:
(12858, 41)


REDUNDANT RAW COLUMN REMOVAL
Dropped: ['StreetNumberNumeric', 'UnparsedAddress', 'SourceFile']


In [11]:

# ============================================================
# 18. Handle missing values
# ============================================================
"""
All learned imputation values are calculated using the training set only.
The same training-set values are then applied to the test set.
"""


# ------------------------------------------------------------
# 18.1 MainLevelBedrooms
# ------------------------------------------------------------
# Missing MainLevelBedrooms is interpreted as no bedroom on the main level.

if "MainLevelBedrooms" in train_df.columns:
    train_df["MainLevelBedrooms"] = (
        train_df["MainLevelBedrooms"]
        .fillna(0)
    )

if "MainLevelBedrooms" in test_df.columns:
    test_df["MainLevelBedrooms"] = (
        test_df["MainLevelBedrooms"]
        .fillna(0)
    )


# ------------------------------------------------------------
# 18.2 Categorical columns
# ------------------------------------------------------------

categorical_unknown_cols = [
    "Flooring",
    "HighSchoolDistrict",
    "MLSAreaMajor",
    "Levels",
    "City",
    "PostalCode"
]

for col in categorical_unknown_cols:

    if col in train_df.columns:
        train_df[col] = (
            train_df[col]
            .astype("object")
            .fillna("Unknown")
        )

    if col in test_df.columns:
        test_df[col] = (
            test_df[col]
            .astype("object")
            .fillna("Unknown")
        )


# ------------------------------------------------------------
# 18.3 Yes/No columns
# ------------------------------------------------------------

yn_cols = [
    "AttachedGarageYN",
    "ViewYN",
    "PoolPrivateYN",
    "NewConstructionYN",
    "FireplaceYN"
]

yn_mapping = {
    "Y": 1,
    "N": 0,
    "Yes": 1,
    "No": 0,
    "YES": 1,
    "NO": 0,
    True: 1,
    False: 0,
    "True": 1,
    "False": 0,
    1: 1,
    0: 0
}

for col in yn_cols:

    if col in train_df.columns:
        train_df[col] = (
            train_df[col]
            .fillna("N")
            .map(yn_mapping)
        )

    if col in test_df.columns:
        test_df[col] = (
            test_df[col]
            .fillna("N")
            .map(yn_mapping)
        )


# ------------------------------------------------------------
# 18.4 Numerical median imputation
# ------------------------------------------------------------

median_cols = [
    "AssociationFee",
    "Stories",
    "GarageSpaces",
    "LotSizeSquareFeet",
    "YearBuilt",
    "LivingArea",
    "BathroomsTotalInteger",
    "ParkingTotal",
    "BedroomsTotal"
]

median_imputation_summary = []

for col in median_cols:

    if col not in train_df.columns:
        continue

    train_df[col] = pd.to_numeric(
        train_df[col],
        errors="coerce"
    )

    if col in test_df.columns:
        test_df[col] = pd.to_numeric(
            test_df[col],
            errors="coerce"
        )

    train_median = train_df[col].median()

    if pd.isna(train_median):
        print(
            f"Warning: {col} has no valid training values. "
            "Median imputation was skipped."
        )
        continue

    train_missing_before = train_df[col].isna().sum()

    test_missing_before = (
        test_df[col].isna().sum()
        if col in test_df.columns
        else 0
    )

    train_df[col] = train_df[col].fillna(
        train_median
    )

    if col in test_df.columns:
        test_df[col] = test_df[col].fillna(
            train_median
        )

    median_imputation_summary.append({
        "Feature": col,
        "TrainingMedian": train_median,
        "TrainValuesImputed": train_missing_before,
        "TestValuesImputed": test_missing_before
    })


median_imputation_summary = pd.DataFrame(
    median_imputation_summary
)

print("\n")
print("=" * 70)
print("MEDIAN IMPUTATION SUMMARY")
print("=" * 70)

display(median_imputation_summary)


# ------------------------------------------------------------
# 18.5 Drop rows missing important location/date fields
# ------------------------------------------------------------

dropna_subset = [
    "Latitude",
    "Longitude",
    "PurchaseContractDate"
]

dropna_subset_train = [
    col for col in dropna_subset
    if col in train_df.columns
]

dropna_subset_test = [
    col for col in dropna_subset
    if col in test_df.columns
]

train_rows_before_dropna = len(train_df)
test_rows_before_dropna = len(test_df)

if dropna_subset_train:
    train_df = train_df.dropna(
        subset=dropna_subset_train
    ).copy()

if dropna_subset_test:
    test_df = test_df.dropna(
        subset=dropna_subset_test
    ).copy()

print("\n")
print("=" * 70)
print("IMPORTANT-FIELD ROW REMOVAL")
print("=" * 70)

print(
    "Training rows removed:",
    train_rows_before_dropna - len(train_df)
)

print(
    "Test rows removed:",
    test_rows_before_dropna - len(test_df)
)






MEDIAN IMPUTATION SUMMARY


,Feature,TrainingMedian,TrainValuesImputed,TestValuesImputed
0,AssociationFee,0.0,17595,3863
1,Stories,1.0,6483,1413
2,GarageSpaces,2.0,2355,485
3,LotSizeSquareFeet,7280.0,1273,241
4,YearBuilt,1977.0,36,12
5,LivingArea,1827.0,49,13
6,BathroomsTotalInteger,2.0,23,8
7,ParkingTotal,2.0,44,6
8,BedroomsTotal,3.0,28,8




IMPORTANT-FIELD ROW REMOVAL
Training rows removed: 35
Test rows removed: 5


In [12]:
# ============================================================
# 19. Drop columns not used by the model
# ============================================================
"""
ClosePrice and Dataset are intentionally retained at this stage.

CloseDate is dropped because the current model does not use raw datetime
values. It could later be transformed into month, year, or days-since features.
"""

drop_before_encoding = [
    "ListingId",
    "ListingKey",
    "ListingKeyNumeric",
    "CloseDate",
    "ContractStatusChangeDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "PropertySubType",
    "PropertyType",
    "MlsStatus",
    "StateOrProvince",
    "Levels",
    "CountyOrParish",
    "City",
    "PostalCode",
    "HighSchoolDistrict",
    "Flooring"
]

train_df = train_df.drop(
    columns=[
        col for col in drop_before_encoding
        if col in train_df.columns
    ]
)

test_df = test_df.drop(
    columns=[
        col for col in drop_before_encoding
        if col in test_df.columns
    ]
)

print("\n")
print("=" * 70)
print("COLUMNS DROPPED BEFORE ENCODING")
print("=" * 70)
print(drop_before_encoding)


# ============================================================
# 20. Final missing-value check before encoding
# ============================================================

train_missing = (
    train_df
    .isna()
    .sum()
)

train_missing = train_missing[
    train_missing > 0
].sort_values(ascending=False)

test_missing = (
    test_df
    .isna()
    .sum()
)

test_missing = test_missing[
    test_missing > 0
].sort_values(ascending=False)

print("\n")
print("=" * 70)
print("MISSING VALUES BEFORE ENCODING")
print("=" * 70)

print("Remaining missing values in train:")
print(train_missing)

print("\nRemaining missing values in test:")
print(test_missing)


# ============================================================
# 21. One-hot encode MLSAreaMajor
# ============================================================

if "MLSAreaMajor" not in train_df.columns:
    raise KeyError(
        "MLSAreaMajor was not found before encoding."
    )

encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False
)

encoded_train_array = encoder.fit_transform(
    train_df[["MLSAreaMajor"]]
)

encoded_test_array = encoder.transform(
    test_df[["MLSAreaMajor"]]
)

encoded_cols = encoder.get_feature_names_out(
    ["MLSAreaMajor"]
)

encoded_train_df = pd.DataFrame(
    encoded_train_array,
    columns=encoded_cols,
    index=train_df.index
)

encoded_test_df = pd.DataFrame(
    encoded_test_array,
    columns=encoded_cols,
    index=test_df.index
)


# Replace MLSAreaMajor with its encoded columns.
train_encoded = pd.concat(
    [
        train_df.drop(columns=["MLSAreaMajor"]),
        encoded_train_df
    ],
    axis=1
)

test_encoded = pd.concat(
    [
        test_df.drop(columns=["MLSAreaMajor"]),
        encoded_test_df
    ],
    axis=1
)


print("\n")
print("=" * 70)
print("MLS AREA ENCODING")
print("=" * 70)

print("Number of encoded MLSAreaMajor columns:", len(encoded_cols))
print("Encoded train shape:", train_encoded.shape)
print("Encoded test shape:", test_encoded.shape)


# ============================================================
# 22. Align train and test columns
# ============================================================

train_encoded, test_encoded = train_encoded.align(
    test_encoded,
    join="left",
    axis=1,
    fill_value=0
)

print("\nTrain and test columns are identical:")
print(train_encoded.columns.equals(test_encoded.columns))


# ============================================================
# 23. Separate model features and target
# ============================================================

y_train = train_encoded["ClosePrice"].copy()
y_test = test_encoded["ClosePrice"].copy()

X_train = train_encoded.drop(
    columns=[
        "ClosePrice",
        "Dataset"
    ],
    errors="ignore"
).copy()

X_test = test_encoded.drop(
    columns=[
        "ClosePrice",
        "Dataset"
    ],
    errors="ignore"
).copy()


# Ensure feature columns are in the same order.
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)


print("\n")
print("=" * 70)
print("FINAL MODEL DATA")
print("=" * 70)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nFeature columns match:")
print(X_train.columns.equals(X_test.columns))


# ============================================================
# 24. Check model feature data types
# ============================================================

object_features = X_train.select_dtypes(
    include=["object", "string", "category", "datetime"]
).columns.tolist()

print("\nNon-numeric columns remaining in X_train:")
print(object_features)

if object_features:
    print(
        "\nWarning: Some non-numeric columns remain. "
        "They must be dropped or encoded before fitting models."
    )


infinite_train_count = (
    np.isinf(
        X_train.select_dtypes(include=[np.number])
    )
    .sum()
    .sum()
)

infinite_test_count = (
    np.isinf(
        X_test.select_dtypes(include=[np.number])
    )
    .sum()
    .sum()
)

print("\nInfinite values in numerical X_train:", infinite_train_count)
print("Infinite values in numerical X_test:", infinite_test_count)


# ============================================================
# 25. Final cleaned-data missing-value check
# ============================================================

final_train_missing = X_train.isna().sum()
final_train_missing = final_train_missing[
    final_train_missing > 0
].sort_values(ascending=False)

final_test_missing = X_test.isna().sum()
final_test_missing = final_test_missing[
    final_test_missing > 0
].sort_values(ascending=False)

print("\n")
print("=" * 70)
print("FINAL FEATURE MISSING-VALUE CHECK")
print("=" * 70)

print("Missing values remaining in X_train:")
print(final_train_missing)

print("\nMissing values remaining in X_test:")
print(final_test_missing)


# ============================================================
# 26. Create final cleaned train/test dataframes for saving
# ============================================================

cleaned_train_df = X_train.copy()
cleaned_train_df["ClosePrice"] = y_train
cleaned_train_df["Dataset"] = "Train"

cleaned_test_df = X_test.copy()
cleaned_test_df["ClosePrice"] = y_test
cleaned_test_df["Dataset"] = "Test"

final_cleaned_df = pd.concat(
    [
        cleaned_train_df,
        cleaned_test_df
    ],
    axis=0,
    ignore_index=True
)


print("\n")
print("=" * 70)
print("FINAL CLEANED DATASET")
print("=" * 70)

print("Final combined shape:", final_cleaned_df.shape)

print("\nDataset distribution:")
print(final_cleaned_df["Dataset"].value_counts())

print("\nFinal ClosePrice summary by dataset:")
display(
    final_cleaned_df.groupby("Dataset")["ClosePrice"].describe()
)



COLUMNS DROPPED BEFORE ENCODING
['ListingId', 'ListingKey', 'ListingKeyNumeric', 'CloseDate', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate', 'PropertySubType', 'PropertyType', 'MlsStatus', 'StateOrProvince', 'Levels', 'CountyOrParish', 'City', 'PostalCode', 'HighSchoolDistrict', 'Flooring']


MISSING VALUES BEFORE ENCODING
Remaining missing values in train:
Series([], dtype: int64)

Remaining missing values in test:
Series([], dtype: int64)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)




MLS AREA ENCODING
Number of encoded MLSAreaMajor columns: 968
Encoded train shape: (61550, 988)
Encoded test shape: (12853, 988)

Train and test columns are identical:
True




FINAL MODEL DATA
X_train shape: (61550, 986)
y_train shape: (61550,)
X_test shape: (12853, 986)
y_test shape: (12853,)

Feature columns match:
True

Non-numeric columns remaining in X_train:
[]

Infinite values in numerical X_train: 0
Infinite values in numerical X_test: 0




FINAL FEATURE MISSING-VALUE CHECK
Missing values remaining in X_train:
Series([], dtype: int64)

Missing values remaining in X_test:
Series([], dtype: int64)




FINAL CLEANED DATASET
Final combined shape: (74403, 988)

Dataset distribution:
Dataset
Train    61550
Test     12853
Name: count, dtype: int64

Final ClosePrice summary by dataset:


,count,mean,std,min,25%,50%,75%,max
Dataset,,,,,,,,
Test,12853.0,1.306029e+06,1.536733e+06,580.0,635000.0,925000.0,1497500.0,46950000.0
Train,61550.0,1.254929e+06,1.356489e+06,85000.0,620000.0,890000.0,1425000.0,60000000.0


In [13]:
# ============================================================
# 27. Save outputs
# ============================================================

final_cleaned_df.to_csv(
    "cleaned_housing_data.csv",
    index=False
)

count_cleaning_summary.to_csv(
    "count_sanity_check_summary.csv",
    index=False
)

removed_target_outliers.to_csv(
    "removed_training_target_outliers.csv",
    index=False
)

print("\nFiles saved:")
print("1. cleaned_housing_data.csv")
print("2. count_sanity_check_summary.csv")
print("3. removed_training_target_outliers.csv")


Files saved:
1. cleaned_housing_data.csv
2. count_sanity_check_summary.csv
3. removed_training_target_outliers.csv
